# Lezione 03 — Versionamento di dati e modelli

**Corso:** MLOps e Monitoraggio con Python · **Blocco 2:** Versionare e misurare
**Tempo stimato:** 100 minuti · **Capitolo del manuale:** `manual/03_versionado_datos_modelos_IT.md`

## Cosa vedrai in questa lezione

- Perché **Git non basta** per versionare dati e modelli (file grandi, binari, che cambiano per intero).
- **DVC** (`dvc init`, `dvc add`) per versionare `datos/churn.csv`, con un **remoto locale** (una cartella, non il cloud).
- Il **linaje** di una versione: da quale codice e da quali dati di origine esce.
- Il principio dei **dati immutabili**: mai sovrascrivere, sempre creare una versione nuova.
- Recuperare una versione precedente con `git checkout` + `dvc checkout`, e **verificarlo davvero** leggendo il file.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata e distrutta da questo stesso notebook: non tocchiamo il repository del corso.

## Prerequisiti

- `pandas` e `dvc` (la cella seguente installa solo ciò che manca, anche in Google Colab). `git` deve essere disponibile nel sistema (lo è in Colab e nella maggior parte delle installazioni).
- La cartella `datos/` del corso, con `churn.csv` all'interno.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta.

---
## Installazione

Installa solo ciò che manca.

In [ ]:
import subprocess, sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("dvc", "dvc")
print("Librerie pronte.")

---
## Cella 0 — Setup

Definiamo `cargar_datos` (per trovare `churn.csv` in `datos/`, `../datos/` o `/content/datos/`) e due funzioni ausiliarie: `ejecutar`, che lancia un comando da riga di comando con `subprocess.run(..., capture_output=True, text=True)`, e `mostrar`, che stampa il suo risultato in modo compatto. Non c'è nessuna fonte di casualità in questo notebook, quindi non serve fissare nessun seme.

In [ ]:
import os
import shutil
import subprocess
import sys

import pandas as pd

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

def ejecutar(comando, cwd):
    """Esegue un comando da riga di comando e restituisce il suo risultato, senza sollevare eccezioni se fallisce."""
    return subprocess.run(comando, cwd=cwd, capture_output=True, text=True)


def mostrar(etiqueta, resultado):
    """Stampa il risultato di un comando in modo compatto, utile per seguire il notebook."""
    print(f"$ {etiqueta}  (rc={resultado.returncode})")
    salida = (resultado.stdout + resultado.stderr).strip()
    if salida:
        for linea in salida.splitlines()[:8]:
            print("   ", linea)

# Risolviamo il percorso dell'eseguibile "dvc": di solito viene installato accanto allo stesso interprete Python
DVC = shutil.which("dvc") or os.path.join(os.path.dirname(sys.executable), "dvc")

# Data di autore/commit fissa: così l'hash di ogni commit è lo stesso a ogni esecuzione (riproducibile)
for variable in ("GIT_AUTHOR_DATE", "GIT_COMMITTER_DATE"):
    os.environ[variable] = "2024-01-01T00:00:00+00:00"

print("Librerie e funzioni ausiliarie pronte.")
print("Eseguibile di dvc:", DVC)

---
## 1. Cartella di lavoro effimera

Creiamo `proyecto_churn/` (e, accanto a essa, `proyecto_churn_remoto/`) da zero a ogni esecuzione, e copiamo `churn.csv` al suo interno: l'originale in `datos/` non si tocca.

In [ ]:
CARPETA = "proyecto_churn"
REMOTO = "proyecto_churn_remoto"

# Puliamo e ricreiamo entrambe le cartelle perché il notebook sia ripetibile da zero
for carpeta in (CARPETA, REMOTO):
    if os.path.exists(carpeta):
        shutil.rmtree(carpeta)
os.makedirs(CARPETA, exist_ok=True)

shutil.copyfile(cargar_datos("churn.csv"), os.path.join(CARPETA, "churn.csv"))
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))
print("churn.csv copiato dentro la cartella effimera (l'originale in datos/ non si tocca).")

---
## 2. `git init` + `dvc init`

Inizializziamo un repository Git vero, con un'identità locale di prova, e poi DVC.

In [ ]:
mostrar("git init", ejecutar(["git", "init"], cwd=CARPETA))
mostrar("git config user.email", ejecutar(["git", "config", "user.email", "curso@ejemplo.local"], cwd=CARPETA))
mostrar("git config user.name", ejecutar(["git", "config", "user.name", "Curso MLOps"], cwd=CARPETA))
mostrar("dvc init", ejecutar([DVC, "init"], cwd=CARPETA))

assert os.path.isdir(os.path.join(CARPETA, ".git"))
assert os.path.isdir(os.path.join(CARPETA, ".dvc"))
print(".git e .dvc inizializzati dentro", CARPETA)

---
## 3. Un remoto locale di DVC

Registriamo `../proyecto_churn_remoto` come remoto predefinito: una cartella, non un servizio nel cloud.

In [ ]:
ruta_remoto = os.path.join("..", REMOTO)
r = ejecutar([DVC, "remote", "add", "-d", "remoto_local", ruta_remoto], cwd=CARPETA)
mostrar("dvc remote add -d remoto_local " + ruta_remoto, r)
assert r.returncode == 0

r = ejecutar([DVC, "remote", "list"], cwd=CARPETA)
mostrar("dvc remote list", r)

---
## 4. Versionare la v1 di `churn.csv`

`dvc add` calcola l'hash e crea `churn.csv.dvc`; lo confermiamo con Git e lo inviamo al remoto con `dvc push`.

In [ ]:
r = ejecutar([DVC, "add", "churn.csv"], cwd=CARPETA)
mostrar("dvc add churn.csv", r)
assert r.returncode == 0
assert os.path.exists(os.path.join(CARPETA, "churn.csv.dvc"))

with open(os.path.join(CARPETA, "churn.csv.dvc"), encoding="utf-8") as f:
    contenido_dvc_v1 = f.read()
print("Contenuto di churn.csv.dvc (v1):")
print(contenido_dvc_v1)

mostrar("git add", ejecutar(["git", "add", "churn.csv.dvc", ".gitignore"], cwd=CARPETA))
mostrar("git commit v1", ejecutar(["git", "commit", "-m", "v1: churn.csv original"], cwd=CARPETA))
mostrar("dvc push", ejecutar([DVC, "push"], cwd=CARPETA))

r = ejecutar(["git", "rev-parse", "HEAD"], cwd=CARPETA)
hash_commit_v1 = r.stdout.strip()
print("Commit della v1:", hash_commit_v1)

**Cosa osservi?** `churn.csv.dvc` è un piccolo file di testo (poche righe) con l'hash MD5 e la dimensione di `churn.csv`: è esattamente ciò che Git può versionare bene. Il file grande vero e proprio è già nel remoto dopo `dvc push`.

---
## 5. Una seconda pulizia reale: la v2

Limitiamo `tickets_soporte` a un massimo di 5 (pochi valori anomali) e versioniamo di nuovo.

In [ ]:
df = pd.read_csv(os.path.join(CARPETA, "churn.csv"))
suma_antes = int(df["tickets_soporte"].sum())
maximo_antes = int(df["tickets_soporte"].max())
filas_afectadas = int((df["tickets_soporte"] > 5).sum())

# Seconda pulizia: limitiamo i ticket di supporto a un massimo di 5 (outlier)
df["tickets_soporte"] = df["tickets_soporte"].clip(upper=5)
df.to_csv(os.path.join(CARPETA, "churn.csv"), index=False)

suma_despues = int(df["tickets_soporte"].sum())
maximo_despues = int(df["tickets_soporte"].max())

print("righe con più di 5 ticket (interessate dal clip):", filas_afectadas)
print("massimo prima/dopo:", maximo_antes, "→", maximo_despues)
print("somma prima/dopo:", suma_antes, "→", suma_despues)
assert maximo_despues == 5 and maximo_antes > maximo_despues

In [ ]:
r = ejecutar([DVC, "add", "churn.csv"], cwd=CARPETA)
mostrar("dvc add churn.csv (v2)", r)
assert r.returncode == 0

with open(os.path.join(CARPETA, "churn.csv.dvc"), encoding="utf-8") as f:
    contenido_dvc_v2 = f.read()
print("Contenuto di churn.csv.dvc (v2):")
print(contenido_dvc_v2)
assert contenido_dvc_v2 != contenido_dvc_v1, "l'hash della v2 deve essere diverso da quello della v1"

mostrar("git add", ejecutar(["git", "add", "churn.csv.dvc"], cwd=CARPETA))
mostrar("git commit v2", ejecutar(["git", "commit", "-m", "v2: tickets_soporte limitado a 5 (outliers)"], cwd=CARPETA))
mostrar("dvc push", ejecutar([DVC, "push"], cwd=CARPETA))

**Cosa osservi?** Il contenuto di `churn.csv.dvc` è cambiato (hash diverso): è una versione nuova, non un aggiornamento della precedente. Il `churn.csv.dvc` della v1 continua a esistere nella cronologia di Git (nel commit precedente), anche se nella cartella di lavoro non lo vedi più.

---
## 6. Il principio dei dati immutabili, in spazio reale

Contiamo quanti oggetti distinti ci sono nel remoto: uno per ogni versione, perché DVC non sovrascrive mai un hash già salvato.

In [ ]:
# Contiamo quanti oggetti distinti conserva il remoto: uno per ogni versione, non vengono mai sovrascritti
objetos_remoto = []
for raiz, _, archivos in os.walk(REMOTO):
    for nombre in archivos:
        objetos_remoto.append(os.path.join(raiz, nombre))

peso_total_kb = sum(os.path.getsize(p) for p in objetos_remoto) / 1024
print("oggetti salvati nel remoto (uno per versione distinta):", len(objetos_remoto))
print("spazio totale occupato nel remoto (KB):", round(peso_total_kb, 1))
assert len(objetos_remoto) == 2, "dovrebbero esistere esattamente 2 versioni salvate (v1 e v2)"

**Cosa osservi?** Due versioni di un file di ≈150 KB occupano nel remoto quasi il doppio dello spazio di una sola: è il prezzo dell'immutabilità. Con dataset grandi e molte versioni, questo costo va tenuto in conto (per esempio, ripulendo versioni molto vecchie dal remoto quando non servono più a nessuno).

---
## 7. Recuperare la v1 con `git checkout` + `dvc checkout`

Prima recuperiamo il puntatore (`.dvc`) del commit della v1, poi sincronizziamo il file reale.

In [ ]:
r = ejecutar(["git", "checkout", hash_commit_v1, "--", "churn.csv.dvc"], cwd=CARPETA)
mostrar("git checkout " + hash_commit_v1[:8] + " -- churn.csv.dvc", r)
assert r.returncode == 0

with open(os.path.join(CARPETA, "churn.csv.dvc"), encoding="utf-8") as f:
    contenido_recuperado = f.read()
assert contenido_recuperado == contenido_dvc_v1, "il puntatore recuperato dovrebbe coincidere con quello della v1"

r = ejecutar([DVC, "checkout", "churn.csv.dvc"], cwd=CARPETA)
mostrar("dvc checkout churn.csv.dvc", r)
assert r.returncode == 0

---
## 8. Verifica concreta: è davvero la v1?

Non basta che i comandi non falliscano: leggiamo il file recuperato e controlliamo una statistica reale.

In [ ]:
# Verifica concreta: leggiamo il file recuperato e controlliamo una statistica reale
df_recuperado = pd.read_csv(os.path.join(CARPETA, "churn.csv"))
maximo_recuperado = int(df_recuperado["tickets_soporte"].max())
suma_recuperada = int(df_recuperado["tickets_soporte"].sum())

print("massimo di tickets_soporte dopo dvc checkout:", maximo_recuperado)
print("somma di tickets_soporte dopo dvc checkout:", suma_recuperada)

assert maximo_recuperado == maximo_antes, "il massimo recuperato dovrebbe essere quello della v1 (non limitato), non quello della v2"
assert suma_recuperada == suma_antes, "la somma recuperata dovrebbe essere quella della v1, non quella della v2"
assert maximo_recuperado != maximo_despues, "se coincidesse con maximo_despues, staremmo ancora vedendo la v2, non la v1"
print("Confermato: il file recuperato è la v1 originale, non la v2 ripulita.")

---
## Riepilogo della lezione

- Git versiona bene il **codice** (il puntatore `.dvc`); DVC versiona il **contenuto grande** (il dataset o il modello reale), appoggiandosi a un hash.
- Un **remoto di DVC** può essere, come in questo corso, una semplice cartella locale: il meccanismo è lo stesso di un remoto nel cloud.
- Il **linaje** di una versione combina il commit di Git (cosa è cambiato, quando) e l'hash di DVC (quali byte esatti).
- Il principio dei **dati immutabili** significa che ogni pulizia crea una versione nuova, non sovrascrive mai la precedente; la conseguenza diretta è più spazio occupato nel remoto.
- Recuperare una versione precedente richiede **due passi** (`git checkout` del puntatore + `dvc checkout` del contenuto), ed è bene verificarlo leggendo il file, non limitandosi a fidarsi che il comando non abbia dato errore.

*Continua con la lezione 4: Tracciamento degli esperimenti.*